Imports

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
import librosa

Merge the two CSV into one

In [26]:
df_1 = pd.read_csv('static_annotations_averaged_songs_1_2000.csv')
df_2 = pd.read_csv('static_annotations_averaged_songs_2000_2058.csv')

df_merged = pd.concat([df_1, df_2], ignore_index=True)

print('df_1_shape:', df_1.shape)
print('df_2 shape:', df_2.shape)
print('Merged shape:', df_merged.shape)

df_merged.head()

df_1_shape: (1744, 5)
df_2 shape: (58, 13)
Merged shape: (1802, 13)


,song_id,valence_mean,valence_std,arousal_mean,arousal_std,valence_ max_mean,valence_max_std,valence_min_mean,valence_min_std,arousal_max_mean,arousal_max_std,arousal_min_mean,arousal_min_std
0,2,3.1,0.94,3.0,0.63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,3.5,1.75,3.3,1.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,5.7,1.42,5.5,1.63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,4.4,2.01,5.3,1.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7,5.8,1.47,6.4,1.69,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# normalize column names and remove unnecessary columns
df_merged.columns = df_merged.columns.str.strip().str.replace(' ', '', regex=False)

columns_to_drop = [
    'valence_max_mean', 'valence_max_std', 'valence_min_mean', 'valence_min_std',
    'arousal_max_mean', 'arousal_max_std', 'arousal_min_mean', 'arousal_min_std'
 ]

existing_columns_to_drop = [col for col in columns_to_drop if col in df_merged.columns]
missing_columns = [col for col in columns_to_drop if col not in df_merged.columns]

df_merged = df_merged.drop(columns=existing_columns_to_drop)

print('Dropped:', existing_columns_to_drop)
print('Not found:', missing_columns)
print('New shape:', df_merged.shape)
df_merged.head()

Dropped: ['valence_max_mean', 'valence_max_std', 'valence_min_mean', 'valence_min_std', 'arousal_max_mean', 'arousal_max_std', 'arousal_min_mean', 'arousal_min_std']
Not found: []
New shape: (1802, 5)


,song_id,valence_mean,valence_std,arousal_mean,arousal_std
0,2,3.1,0.94,3.0,0.63
1,3,3.5,1.75,3.3,1.62
2,4,5.7,1.42,5.5,1.63
3,5,4.4,2.01,5.3,1.85
4,7,5.8,1.47,6.4,1.69


Audio data Preparation

In [28]:
audio_dir = Path('MEMD_audio')

data = df_merged.copy()

data['audio_path'] = data['song_id'].apply(lambda x: audio_dir / f'{x}.mp3')

data['audio_exists'] = data['audio_path'].apply(lambda p: Path(p).exists())
data = data[data['audio_exists']].copy()

data = data[['song_id', 'audio_path', 'valence_mean', 'arousal_mean']].copy()
data = data.rename(columns={
    'valence_mean': 'valence',
    'arousal_mean': 'arousal'
})

In [29]:
print(data.head())
print(f'Shape: {data.shape}')
print('Check duplicates: ', data['song_id'].duplicated().sum())
print('Check for missing values: \n', data[['valence', 'arousal']].isna().sum())

   song_id        audio_path  valence  arousal
0        2  MEMD_audio/2.mp3      3.1      3.0
1        3  MEMD_audio/3.mp3      3.5      3.3
2        4  MEMD_audio/4.mp3      5.7      5.5
3        5  MEMD_audio/5.mp3      4.4      5.3
4        7  MEMD_audio/7.mp3      5.8      6.4
Shape: (1802, 4)
Check duplicates:  0
Check for missing values: 
 valence    0
arousal    0
dtype: int64


Split data in 70% train 15% val 15% test

In [30]:
train_data, temp_data = train_test_split(data, test_size=0.30, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.50, random_state=42)

train_data = train_data.copy()
val_data = val_data.copy()
test_data = test_data.copy()

train_data['split'] = 'train'
val_data['split'] = 'val'
test_data['split'] = 'test'

final_df = pd.concat([train_data, val_data, test_data], ignore_index=True)

print(final_df['split'].value_counts())

split
train    1261
test      271
val       270
Name: count, dtype: int64


Save the final dataframe

In [31]:
final_df.to_csv('deam_static_splits.csv', index=False)

Beginning of a baseline model to test and later change.

audio -> mono
resmaple -> 16000
crop length -> fixed crop length: 20 secs
form -> log-mel spectrogram
regression target -> [valence, arousal]

In [32]:
class DEAMDataset(Dataset):
    def __init__(
            self,
            dataframe,
            sample_rate=16000,
            duration=20,
            n_mels=128,
            n_fft=1024,
            hop_length=512,
            train=True
    ):
        self.df = dataframe.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.num_samples = sample_rate * duration
        self.train = train

        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)
    
    # optional: implement audio loading with torchaudio or librosa depending on your where is running

    #def _load_audio(self, path):
     #   waveform, sr = torchaudio.load(path)

        # conversion to mono
      #  if waveform.shape[0] > 1:
        #    waveform = waveform.mean(dim=0, keepdim=True)

        # resample if needed
        #if sr != self.sample_rate:
        #    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.sample_rate)
        #    waveform = resampler(waveform)

       # return waveform
    def _load_audio(self, path):
        waveform_np, sr = librosa.load(path, sr=self.sample_rate, mono=True)
        waveform = torch.tensor(waveform_np, dtype=torch.float32).unsqueeze(0)
        return waveform

    def _crop_or_pad(self, waveform):
        length = waveform.shape[1]

        if length > self.num_samples:
            if self.train:
                start = torch.randint(0, length - self .num_samples + 1, (1,)).item()
            else:
                start = (length - self.num_samples) // 2
            waveform = waveform[:, start:start + self.num_samples]
        elif length < self.num_samples:
            pad_amount = self.num_samples - length
            waveform = F.pad(waveform, (0, pad_amount))

        return waveform
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        waveform = self._load_audio(str(row['audio_path']))
        waveform = self._crop_or_pad(waveform)

        mel = self.mel_transform(waveform) # [1, n_mels, time_frames]
        mel = self.db_transform(mel) # log scale

        # optional normalization
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        target = torch.tensor([row['valence'], row['arousal']], dtype=torch.float32)

        return mel, target

Dataloaders

In [33]:
# ATTENTION: This cell is computationally intensive and probaly does not work on laptops without GPU and sufficient memory
# Probably another version might exist to sanity check the model
# Run the other cell instead on the laptop and run this one on colab
# The test version exists in the end of the notebook


train_dataset = DEAMDataset(train_data, train=True)
val_dataset = DEAMDataset(val_data, train=False)
test_dataset = DEAMDataset(test_data, train=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

CNN

In [34]:
class SimpleAudioCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

Set training

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleAudioCNN().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Train and validation

In [36]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            preds = model(xb)
            loss = criterion(preds, yb)

            total_loss += loss.item() * xb.size(0)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(yb.cpu().numpy())

        all_preds = np.vstack(all_preds)
        all_targets = np.vstack(all_targets)

        val_rmse = np.sqrt(np.mean((all_preds[:, 0] - all_targets[:, 0]) ** 2))
        aro_rmse = np.sqrt(np.mean((all_preds[:, 1] - all_targets[:, 1]) ** 2))

        try:
            val_pearson = pearsonr(all_preds[:, 0], all_targets[:, 0])[0]
        except:
            val_pearson = np.nan

        try:
            aro_pearson = pearsonr(all_preds[:, 1], all_targets[:, 1])[0]
        except:
            aro_pearson = np.nan

        avg_loss = total_loss / len(loader.dataset)

        return{
            'loss': avg_loss,
            'valence_rmse': val_rmse,
            'arousal_rmse': aro_rmse,
            'valence_pearson': val_pearson,
            'arousal_pearson': aro_pearson,
            'preds': all_preds,
            'targets': all_targets
        }

### Metric Note: Pearson Correlation

`valence_pearson` and `arousal_pearson` are Pearson correlation coefficients (`r`) between predictions and true targets.

Pearson formula:

$$
r = \frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}}
$$

- `r = 1.0`: perfect positive linear agreement
- `r = 0.0`: no linear relationship
- `r = -1.0`: perfect negative linear relationship

Use Pearson together with RMSE:
- RMSE tells you absolute prediction error magnitude.
- Pearson tells you whether the model captures the trend/ranking of targets.

A model can have reasonable Pearson but still poor RMSE if predictions are biased in scale or offset.

Training loop

In [ ]:
# CAUTION do not run this cell on the laptop, use the test version at the end

best_val_loss = float('inf')
patience = 5
patience_counter = 0
num_epochs = 30

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    print(
        f'Epoch {epoch+1:02d} | '
        f'Train Loss: {train_loss:.4f} | '
        f'Val Loss: {val_metrics["loss"]:.4f} | '
        f'Valence RMSE: {val_metrics["valence_rmse"]:.4f} | '
        f'Arousal RMSE: {val_metrics["arousal_rmse"]:.4f} | '
        f'Valence Pearson: {val_metrics["valence_pearson"]:.4f} | '
        f'Arousal Pearson: {val_metrics["arousal_pearson"]:.4f}'
    )
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        patience_counter = 0
        torch.save(model.state_dict(), 'best_deam_cnn.pth')
        print('Saved best model.')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print('Early stopping triggered')
            break

Test best model

In [ ]:
model.load_state_dict(torch.load('best_deam_cnn.pth', map_location=device))

test_metrics = evaluate(model, test_loader, criterion, device)

print(
    f'Test Loss: {test_metrics["loss"]:.4f} | '
    f'Valence RMSE: {test_metrics["valence_rmse"]:.4f} | '
    f'Arousal RMSE: {test_metrics["arousal_rmse"]:.4f} | '
    f'Valence Pearson: {test_metrics["valence_pearson"]:.4f} | '
    f'Arousal Pearson: {test_metrics["arousal_pearson"]:.4f}'
)

Save predictions

In [ ]:
test_preds = test_metrics['preds']
test_targets = test_metrics['targets']

results_df = test_data.reset_index(drop=True).copy()
results_df['pred_valence'] = test_preds[:, 0]
results_df['pred_arousal'] = test_preds[:, 1]
results_df['true_valence'] = test_targets[:, 0]
results_df['true_arousal'] = test_targets[:, 1]

results_df.to_csv('deam_test_results.csv', index=False)
print(results_df.head())

TEST BEFORE BEING MOVED

ONLY TO RUN ON THE LAPTOP

In [37]:
mini_train_n = 16
mini_val_n = 8
mini_test_n = 8

mini_train_data = train_data.sample(n=min(mini_train_n, len(train_data)), random_state=42).reset_index(drop=True)
mini_val_data = val_data.sample(n=min(mini_val_n, len(val_data)), random_state=42).reset_index(drop=True)
mini_test_data = test_data.sample(n=min(mini_test_n, len(test_data)), random_state=42).reset_index(drop=True)

print('Mini split sizes:')
print(f'Train: {len(mini_train_data)}')
print(f'Val: {len(mini_val_data)}')
print(f'Test: {len(mini_test_data)}')

Mini split sizes:
Train: 16
Val: 8
Test: 8


In [38]:
mini_train_dataset = DEAMDataset(
    mini_train_data,
    sample_rate=16000,
    duration=3,
    n_mels=64,
    n_fft=512,
    hop_length=256,
    train=True
)

mini_val_dataset = DEAMDataset(
    mini_val_data,
    sample_rate=16000,
    duration=3,
    n_mels=64,
    n_fft=512,
    hop_length=256,
    train=False
)

mini_test_dataset = DEAMDataset(
    mini_test_data,
    sample_rate=16000,
    duration=3,
    n_mels=64,
    n_fft=512,
    hop_length=256,
    train=False
)

mini_train_loader = DataLoader(mini_train_dataset, batch_size=4, shuffle=True, num_workers=0)
mini_val_loader = DataLoader(mini_val_dataset, batch_size=4, shuffle=False, num_workers=0)
mini_test_loader = DataLoader(mini_test_dataset, batch_size=4, shuffle=False, num_workers=0)

print('Mini loaders created')


Mini loaders created


In [39]:
mini_device = torch.device('cpu')
mini_model = SimpleAudioCNN().to(mini_device)
mini_criterion = nn.MSELoss()
mini_optimizer = torch.optim.Adam(mini_model.parameters(), lr=1e-3)

xb, yb = next(iter(mini_train_loader))
print('Input batch shape:', xb.shape)
print('Target batch shape:', yb.shape)

xb = xb.to(mini_device)
yb = yb.to(mini_device)

with torch.no_grad():
    preds = mini_model(xb)

print('Predictions shape:', preds.shape)

loss = mini_criterion(preds, yb)
print('Initial loss:', loss.item())

Input batch shape: torch.Size([4, 1, 64, 188])
Target batch shape: torch.Size([4, 2])
Predictions shape: torch.Size([4, 2])
Initial loss: 23.1943359375


In [40]:
mini_epochs = 2
best_mini_val_loss = float('inf')

for epoch in range(mini_epochs):
    train_loss = train_one_epoch(
        mini_model, mini_train_loader, mini_optimizer, mini_criterion, mini_device
    )

    val_metrics = evaluate(mini_model, mini_val_loader, mini_criterion, mini_device)

    print(
        f'Mini Epoch {epoch+1:02d} | '
        f'Train Loss: {train_loss:.4f} | '
        f'Val Loss: {val_metrics["loss"]:.4f} | '
        f'Valence RMSE: {val_metrics["valence_rmse"]:.4f} | '
        f'Arousal RMSE: {val_metrics["arousal_rmse"]:.4f} | '
    )

    if val_metrics['loss'] < best_mini_val_loss:
        best_mini_val_loss = val_metrics['loss']
        torch.save(mini_model.state_dict(), 'best_mini_deam_cnn.pth')
        print('Saved best mini model.')

print('Mini test finshed')

Mini Epoch 01 | Train Loss: 20.2545 | Val Loss: 22.8321 | Valence RMSE: 4.5704 | Arousal RMSE: 4.9776 | 
Saved best mini model.
Mini Epoch 02 | Train Loss: 17.8067 | Val Loss: 21.4901 | Valence RMSE: 4.4334 | Arousal RMSE: 4.8296 | 
Saved best mini model.
Mini test finshed


In [41]:
mini_model.load_state_dict(torch.load('best_mini_deam_cnn.pth', map_location=mini_device))

mini_test_metrics = evaluate(
    mini_model, mini_test_loader, mini_criterion, mini_device
)

print(
    f'Mini Test Loss: {mini_test_metrics["loss"]:.4f} | '
    f'Valence RMSE: {mini_test_metrics["valence_rmse"]:.4f} | '
    f'Arousal RMSE: {mini_test_metrics["arousal_rmse"]:.4f} | '
    f'Valence Pearson: {mini_test_metrics["valence_pearson"]:.4f} | '
    f'Arousal Pearson: {mini_test_metrics["arousal_pearson"]:.4f}'
)

Mini Test Loss: 18.4894 | Valence RMSE: 4.3896 | Arousal RMSE: 4.2084 | Valence Pearson: -0.8297 | Arousal Pearson: -0.8107
